In [1]:
import sys
import os
from pathlib import Path

current = Path.cwd().resolve()
for parent in [current, *current.parents]:
	if (parent / "bots").exists():
		sys.path.append(parent)
		os.chdir(parent)
		break

import numpy as np
import math
os.environ.pop("DEBUG", None)
os.environ["BEAM"] = "2"
from tinygrad import Tensor, Device, TinyJit
from tinygrad.nn.optim import AdamW
from tinygrad.nn.state import get_parameters
from engine.game import GameBatch, Game, GameResult, Color, MAX_MOVES
from models.v0 import Model, Config, init_weights
from helpers.evaluator import Evaluator, Encoding
from helpers.replay_buffer import ReplayBuffer

GAME_BATCH_SIZE = 128
MODEL_BATCH_SIZE = 128
BUFFER_CAPACITY = 200_000

rng = np.random.default_rng(42)	

In [3]:
def sigma(q, max_N):
	return (50.0 + max_N) * q

# IDEALLY, USE TINYGRAD GPU INSTEAD!
def np_softmax(x: np.ndarray, axis=-1):
	x_max = np.max(x, axis=axis, keepdims=True)
	exp_x = np.exp(x - x_max)
	return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def _terminal_value(result): 
	return -1.0 if result == GameResult.CHECKMATE else 0.0

class Node:
	__slots__ = (
		"n_moves", "terminal", "expanded", "children",
		"value", "p_logits", "N", "W"
	)

	def __init__(self, 
		n_moves: int, 
		terminal: bool = False, 
		value: float = 0.0
	):
		self.n_moves = n_moves
		self.terminal = terminal
		self.value = value
		self.expanded: bool = False
		self.p_logits: np.ndarray
		self.N: np.ndarray = None
		self.W: np.ndarray = None
		self.children: list[Node] = None
	
	def expand(self, priors, value):
		self.p_logits = priors
		self.value = value
		self.N = np.zeros(self.n_moves, dtype=np.float32)
		self.W = np.zeros(self.n_moves, dtype=np.float32)
		self.children = [None for _ in range(self.n_moves)]
		self.expanded = True
	
	def Q(self, a):
		return self.W[a] / self.N[a]
	# definitely needs to be improved with numpy / tinygrad GPU vectorization
	def v_mix(self):
		visited = self.N > 0
		if not visited.any():
			return self.value
		policy = np_softmax(self.p_logits)
		w = policy[visited]
		q = self.W[visited] / self.N[visited]
		policy_mean_value = (w * q).sum() / w.sum()
		node_N = self.N.sum()
		return (self.value + node_N * policy_mean_value) / (1 + node_N)
	
	def completed_Q(self):
		_v_mix = self.v_mix()
		return np.where(self.N > 0, self.W / np.maximum(self.N, 1.0), _v_mix)
	
	def improved_policy(self) -> np.ndarray:
		max_N = self.N.max() if self.expanded else 0
		return np_softmax(self.p_logits + sigma(self.completed_Q(), max_N))
	
	def select(self):
		_improved_policy = self.improved_policy()
		node_N: int = self.N.sum()
		return int((_improved_policy - self.N / (1 + node_N)).argmax())
		# return max(
		# 	range(self.n_moves),
		# 	key=lambda a: _improved_policy[a] - self.N[a] / (1 + node_N)
		# )

def make_child(game: Game, parent: Node, a: int):
	game.play(a)
	child = parent.children[a]
	if child is None:
		result = game.result()
		term = result != GameResult.ONGOING
		child = Node(
			game.num_moves(), 
			terminal=term, 
			value=_terminal_value(result) if term else 0.0
		)
		parent.children[a] = child
	return child

def descend(game: Game, root: Node, initial_a: int):
	path = [(root, initial_a)]
	node = make_child(game, root, initial_a)
	while node.expanded:
		a = node.select()
		path.append((node, a))
		node = make_child(game, node, a)
	return node, path

def run_gumbel(
	game_batch: GameBatch,
	evaluator: Evaluator,
	rng: np.random.Generator,
	n_sims: int, 
	k: int
):
	B = len(game_batch)
	active = game_batch.active
	state = Encoding(*[val.copy() for val in game_batch.get_encoding()])

	root_p_logits, root_values = evaluator.eval_logits(Encoding(
		*game_batch.get_encoding()
	))
	n_moves = game_batch.num_moves_all()

	roots: list[Node] = [None for _ in range(B)]
	gumbel = [None for _ in range(B)]
	candidates = [None for _ in range(B)]

	# gumbel noise selection
	for i in range(B):
		if not active[i]: continue
		_n_moves = int(n_moves[i])
		node = Node(_n_moves, terminal=False, value=float(root_values[i]))
		node.expand(
			root_p_logits[i, :_n_moves].astype(np.float32).copy(), 
			float(root_values[i])
		)
		roots[i] = node

		gumbel_noise = rng.gumbel(size=_n_moves).astype(np.float32)
		gumbel[i] = gumbel_noise
		n_selected_moves = min(k, _n_moves)
		candidates[i] = list(
			np.argsort(-(node.p_logits + gumbel_noise))[:n_selected_moves]
		)
	
	# sequential halving
	num_phases = max(1, math.ceil(math.log2(k)))
	for phase in range(num_phases):
		# create what nodes will be evaluated (each item is an initial action idx)
		to_eval = [[] for _ in range(B)]
		max_eval_len = 0
		for i in range(B):
			if not active[i] or len(candidates[i]) <= 1: continue
			sims_per_cand = max(1, 
				n_sims // (num_phases * len(candidates[i]))
			)
			# each action gets sims_per_cand evaluations in in to_eval
			to_eval[i] = [a for a in candidates[i] for _ in range(sims_per_cand)]
			max_eval_len = max(max_eval_len, len(to_eval[i]))

		# run scheduled sims together across games
		for eval_run in range(max_eval_len):
			# find leaves to evaluate
			leaves: list[tuple[int, Node, list[tuple[Node, int]]]] = []
			for i in range(B):
				if eval_run >= len(to_eval[i]): continue
				leaf, path = descend(game_batch[i], roots[i], to_eval[i][eval_run])
				leaves.append((i, leaf, path))
			if not leaves: continue
			p_logits, values = evaluator.eval_logits(Encoding(
				*game_batch.get_encoding()
			))

			for i, leaf, path in leaves:
				if not leaf.terminal:
					_n_moves = leaf.n_moves
					leaf.expand(
						p_logits[i, :_n_moves].astype(np.float32).copy(),
						float(values[i])
					)
				v = leaf.value
				# backup value
				for node, a in reversed(path):
					v = -v
					node.N[a] += 1
					node.W[a] += v
				for _ in range(len(path)):
					game_batch[i].undo()
		
		# elimate bottom half
		for i in range(B):
			if not active[i] or len(candidates[i]) <= 1: continue
			root, gumbel_noise = roots[i], gumbel[i]
			root_max_N = root.N.max()
			ranked = sorted(candidates[i], 
				key=lambda a: 
					gumbel_noise[a] 
					+ root.p_logits[a] 
					+ sigma(root.Q(a) if root.N[a] > 0 else root.v_mix(), root_max_N),
				reverse=True
			)
			candidates[i] = ranked[:max(1, len(ranked) // 2)]
	
	policy = np.zeros((B, MAX_ENGINE_MOVES), dtype=np.float32)
	moves = np.zeros(B, dtype=np.int32)
	for i in range(B):
		if not active[i]: continue
		moves[i] = int(candidates[i][0])
		_n_moves = roots[i].n_moves
		policy[i, :_n_moves] = roots[i].improved_policy()
	return state, policy, moves

In [6]:
batch = GameBatch(GAME_BATCH_SIZE)
config = Config(d_hidden=192, n_heads=4, n_layers=3)
model = Model(config)
# init_weights(model, config)
from tinygrad.nn.state import safe_load, load_state_dict
load_state_dict(model, safe_load("../data/overnight-gumbel/model_003000.safetensors"))

evaluator = Evaluator(model, MODEL_BATCH_SIZE)
replay_buffer = ReplayBuffer(BUFFER_CAPACITY)
traj   = [[] for _ in range(GAME_BATCH_SIZE)]

loaded weights in 266.11 ms, 0.01 GB loaded at 0.02 GB/s


In [4]:
opt = AdamW(get_parameters(model), lr=3e-4)
step = 0
MB = 256
@TinyJit
def train_step(board, castling, ep, rep, clock, moves, num_moves, pi, z):
	p, v = model(board, castling, ep, rep, clock, moves, num_moves)
	policy_loss = -(pi * p.log_softmax(axis=-1)).sum(axis=-1).mean()
	value_loss = (pow(v.squeeze(-1) - z, 2)).mean()
	loss = policy_loss + value_loss
	opt.zero_grad()
	loss.backward()
	opt.step()
	return loss.realize()

In [5]:
while step < 50_000:
	# ---- one self-play ply across all B games ----
	Tensor.training = False
	states, pi, mv = run_gumbel(batch, evaluator, rng, n_sims=32, k=8)
	side = batch.to_moves()
	for i in range(GAME_BATCH_SIZE):                 # record the position we just searched
		traj[i].append((states[i], pi[i].copy(), int(side[i])))

	results = batch.play_batch(mv)
	loser   = batch.to_moves()         # side to move AFTER the move = mated side on checkmate

	for i in range(GAME_BATCH_SIZE):                 # label + flush + restart finished games
		if results[i] == GameResult.ONGOING:
			continue
		if results[i] == GameResult.CHECKMATE:
			winner = 1 - int(loser[i])
			zs = [1.0 if s == winner else -1.0 for (_, _, s) in traj[i]]
		else:                          # stalemate / 50-move / repetition
			zs = [0.0] * len(traj[i])
		for (enc, p_, _), zz in zip(traj[i], zs):
			replay_buffer.add(enc, p_, zz)
		traj[i] = []
		batch[i].reset()
	# ---- one training step once warmed up ----
	if replay_buffer.size >= 1000:
		Tensor.training = True
		s_enc, s_pi, s_z = replay_buffer.sample(MB, rng)
		loss = train_step(*s_enc.tensors(), Tensor(s_pi), Tensor(s_z))
		if step % 25 == 0:
			print(f"step {step}  buffer {replay_buffer.size}  loss {loss.item():.3f}")
	elif step % 25 == 0:
		print(f"loading buffer... {replay_buffer.size}/1000")
	step += 1

loading buffer... 0/1000
loading buffer... 55/1000
loading buffer... 263/1000
loading buffer... 679/1000
step 100  buffer 1267  loss 4.379
step 125  buffer 2403  loss 4.398
step 150  buffer 4691  loss 4.280
step 175  buffer 6011  loss 4.090
step 200  buffer 7687  loss 4.053
step 225  buffer 9729  loss 3.876
step 250  buffer 11533  loss 3.728
step 275  buffer 15030  loss 3.558
step 300  buffer 18731  loss 3.451
step 325  buffer 21515  loss 3.359
step 350  buffer 23239  loss 3.222
step 375  buffer 28255  loss 3.192
step 400  buffer 31282  loss 3.261
step 425  buffer 36991  loss 2.966
step 450  buffer 42064  loss 2.955
step 475  buffer 45998  loss 3.044
step 500  buffer 48965  loss 2.998
step 525  buffer 52796  loss 2.902
step 550  buffer 55393  loss 2.865
step 575  buffer 57805  loss 2.952
step 600  buffer 61135  loss 2.875


KeyboardInterrupt: 

In [ ]:
import time
from engine.game import Color, GameResult

def _random_moves(eb, active, rng):
	nm = eb.num_moves_all()
	return np.array([
    rng.integers(0, int(nm[i])) if (active[i] and nm[i] > 0) else 0
		for i in range(len(eb))
  ], dtype=np.int32)

def eval_vs_random(evaluator, rng, n_games=128, max_plies=200, model_color=Color.WHITE):
	Tensor.training = False
	gbs = GameBatch(n_games)
	for t in range(max_plies):
		active = gbs.active
		if not active.any(): break
		if (t & 1) == int(model_color):
			_, _, mv = run_gumbel(gbs, evaluator, rng, n_sims=16, k=8)
		else:
			mv = _random_moves(gbs, active, rng)
		gbs.play_batch(mv, mask=active)
		print(t, active.sum())

	res, to_move = gbs.results(), gbs.to_moves()
	w = l = d = u = 0
	for i in range(n_games):
			r = res[i]
			if r == GameResult.ONGOING: u += 1                    # hit ply cap
			elif r == GameResult.CHECKMATE:
				# side to move at a checkmate is the mated (losing) side
				if int(to_move[i]) != int(model_color): w += 1
				else: l += 1
			else: d += 1
	return w, l, d, u

# run as white and as black so first-move advantage doesn't flatter the score
ww = eval_vs_random(evaluator, rng, n_games=GAME_BATCH_SIZE, model_color=Color.WHITE)
bb = eval_vs_random(evaluator, rng, n_games=GAME_BATCH_SIZE, model_color=Color.BLACK)
w, l, d, u = (a + b for a, b in zip(ww, bb))
n = 256
print(f"greedy model vs random ({n} games): W {w}  L {l}  D {d}  cap {u}")
print(f"  score {(w + 0.5*(d + u)) / n:.3f}")     # 1=win, 0.5=draw/cap

IndexError: shape mismatch: objects cannot be broadcast to a single shape ((128, 322, 192), (128, 162, 1))